In [1]:
from data_preparation import prepare_dataset, get_loss_weights
from graph_construction import build_graph
from train_and_evaluate import (
    train_and_evaluate,
    train_mlp,
    train_logistic_regression,
    train_random_forest
)
from models import (
    Transformer_Model,
    RGCN_Model,
    GAT_Model,
    BaselineMLP
)
from resources import (
    DATA_PATH,
    DEVICE,
    OUTPUT_TABLES_PATH,
    OUTPUT_FIGURES_PATH
)

EPOCHS = 500
PATIENCE = 30


c:\Users\Mislav.FERIT-PC\Desktop\multirelational-graph-topology-for-downstream-wildfire-tasks\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
import torch
import os
import numpy as np

In [3]:
x, y, pos_combined, pos_spatial, pos_temporal, group_ids = prepare_dataset(DATA_PATH, return_group_ids=True)
class_weights = get_loss_weights(y = y)

There are 10034 natural fires (56.44 %) [all]
There are 7744 human made fires (43.56 %) [all]


# STRUCTURAL ABLATION

In [4]:
NUM_RUNS = 10

## Training with LogArea as only node feature - trying to isolate topology

In [5]:

graph_types = ['spatial', 'temporal', 'combined', 'multirelational']
k_values = [5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35]
architectures = ['RGCN', 'GAT', 'TransformerConv']

results = []

print(f"\nStarting Comprehensive Ablation Study ({NUM_RUNS} runs per config)...")
for g_type in graph_types:
    for k in k_values:
        data = build_graph(g_type, k, pos_spatial, pos_temporal, pos_combined, x, y, group_ids=group_ids)
        in_dim = data.x.size(1) 
        edge_dim = data.edge_attr.size(1)

        for arch in architectures:
            run_f1s = []
            run_accs = []
            
            for run in range(NUM_RUNS):
                if arch == 'RGCN':
                    model = RGCN_Model(input_dim=in_dim, hidden_dim=16)
                elif arch == 'GAT':
                    model = GAT_Model(input_dim=in_dim, hidden_dim=16, edge_dim=edge_dim)
                elif arch == 'TransformerConv':
                    model = Transformer_Model(input_dim=in_dim, hidden_dim=16, edge_dim=edge_dim)

                f1, acc, _ = train_and_evaluate(model, data, DEVICE, class_weights, EPOCHS, PATIENCE)
                run_f1s.append(f1)
                run_accs.append(acc)

            avg_f1, std_f1 = np.mean(run_f1s), np.std(run_f1s)
            avg_acc, std_acc = np.mean(run_accs), np.std(run_accs)

            print(f"Graph: {g_type.upper():<10} | K: {k:<2} | Model: {arch:<15} | F1: {avg_f1:.4f} ± {std_f1:.4f}")
            
            results.append({
                'Graph': g_type,
                'K': k,
                'Model': arch,
                'F1': avg_f1,
                'F1_std': std_f1,
                'Accuracy': avg_acc,
                'Accuracy_std': std_acc
            })

topology_comparison_df = pd.DataFrame(results)
topology_comparison_df.to_csv(os.path.join(OUTPUT_TABLES_PATH, "topology_comparison_df.csv"))


Starting Comprehensive Ablation Study (10 runs per config)...
Graph: SPATIAL    | K: 5  | Model: RGCN            | F1: 0.5663 ± 0.0196
Graph: SPATIAL    | K: 5  | Model: GAT             | F1: 0.5694 ± 0.0017
Graph: SPATIAL    | K: 5  | Model: TransformerConv | F1: 0.5751 ± 0.0110
Graph: SPATIAL    | K: 7  | Model: RGCN            | F1: 0.5477 ± 0.0341
Graph: SPATIAL    | K: 7  | Model: GAT             | F1: 0.5719 ± 0.0070
Graph: SPATIAL    | K: 7  | Model: TransformerConv | F1: 0.5733 ± 0.0154
Graph: SPATIAL    | K: 9  | Model: RGCN            | F1: 0.5633 ± 0.0310
Graph: SPATIAL    | K: 9  | Model: GAT             | F1: 0.5483 ± 0.0518
Graph: SPATIAL    | K: 9  | Model: TransformerConv | F1: 0.5855 ± 0.0098
Graph: SPATIAL    | K: 11 | Model: RGCN            | F1: 0.5598 ± 0.0348
Graph: SPATIAL    | K: 11 | Model: GAT             | F1: 0.5646 ± 0.0121
Graph: SPATIAL    | K: 11 | Model: TransformerConv | F1: 0.5798 ± 0.0401
Graph: SPATIAL    | K: 13 | Model: RGCN            | F1: 0.55

## Training Baselines: MLP, Logistic Regression, and Random Forest (spatial, temporal and combined)


In [6]:
HIDDEN_UNITS = 16

train_mask = data.train_mask.to(DEVICE)
val_mask = data.val_mask.to(DEVICE)
test_mask = data.test_mask.to(DEVICE)

x_spatial = torch.cat([x, pos_spatial], dim=1)                 # [LogArea, Lat, Lon]
x_temporal = torch.cat([x, pos_temporal], dim=1)               # [LogArea, Days]
x_combined = torch.cat([x, pos_spatial, pos_temporal], dim=1) # [LogArea, Lat, Lon, Days]

baseline_feature_configs = [
    ('Spatial', x_spatial),
    ('Temporal', x_temporal),
    ('Combined', x_combined)
]

baseline_results = []

# --- 1. MLP Baselines ---
print("\n--- Running MLP Baselines (" + str(NUM_RUNS) + " runs each) ---")
for feat_name, input_tensor in baseline_feature_configs:
    in_features = input_tensor.size(1)
    run_f1s = []
    run_accs = []
    for run in range(NUM_RUNS):
        model = BaselineMLP(in_features, HIDDEN_UNITS).to(DEVICE)
        f1, acc, _ = train_mlp(model, input_tensor, y, train_mask, val_mask, test_mask, DEVICE, class_weights, epochs=EPOCHS)
        run_f1s.append(f1)
        run_accs.append(acc)
    
    avg_f1, std_f1 = np.mean(run_f1s), np.std(run_f1s)
    avg_acc, std_acc = np.mean(run_accs), np.std(run_accs)
    print(f"Model: {'MLP':<20} | Feature: {feat_name:<10} | F1: {avg_f1:.4f} ± {std_f1:.4f}")
    
    baseline_results.append({
        'Graph': f'MLP_{feat_name}',
        'K': 0,
        'Model': 'MLP',
        'F1': avg_f1,
        'F1_std': std_f1,
        'Accuracy': avg_acc,
        'Accuracy_std': std_acc
    })

# --- 2. Logistic Regression Baselines ---
print("\n--- Running Logistic Regression Baselines (" + str(NUM_RUNS) + " runs each) ---")
for feat_name, input_tensor in baseline_feature_configs:
    run_f1s = []
    run_accs = []
    for run in range(NUM_RUNS):
        f1, acc, _ = train_logistic_regression(
            input_tensor, y, train_mask, test_mask, random_state=run, max_iter=1000
        )
        run_f1s.append(f1)
        run_accs.append(acc)
    
    avg_f1, std_f1 = np.mean(run_f1s), np.std(run_f1s)
    avg_acc, std_acc = np.mean(run_accs), np.std(run_accs)
    print(f"Model: {'Logistic Regression':<20} | Feature: {feat_name:<10} | F1: {avg_f1:.4f} ± {std_f1:.4f}")
    
    baseline_results.append({
        'Graph': f'LR_{feat_name}',
        'K': 0,
        'Model': 'Logistic Regression',
        'F1': avg_f1,
        'F1_std': std_f1,
        'Accuracy': avg_acc,
        'Accuracy_std': std_acc
    })

# --- 3. Random Forest Baselines ---
print("\n--- Running Random Forest Baselines (" + str(NUM_RUNS) + " runs each) ---")
for feat_name, input_tensor in baseline_feature_configs:
    run_f1s = []
    run_accs = []
    for run in range(NUM_RUNS):
        f1, acc, _ = train_random_forest(
            input_tensor, y, train_mask, test_mask, n_estimators=100, random_state=run, n_jobs=-1
        )
        run_f1s.append(f1)
        run_accs.append(acc)
    
    avg_f1, std_f1 = np.mean(run_f1s), np.std(run_f1s)
    avg_acc, std_acc = np.mean(run_accs), np.std(run_accs)
    print(f"Model: {'Random Forest':<20} | Feature: {feat_name:<10} | F1: {avg_f1:.4f} ± {std_f1:.4f}")
    
    baseline_results.append({
        'Graph': f'RF_{feat_name}',
        'K': 0,
        'Model': 'Random Forest',
        'F1': avg_f1,
        'F1_std': std_f1,
        'Accuracy': avg_acc,
        'Accuracy_std': std_acc
    })

results_baselines_df = pd.DataFrame(baseline_results)
mlp_topology_comparison_df = pd.concat([topology_comparison_df, results_baselines_df], ignore_index=True)

mlp_topology_comparison_df.to_csv(os.path.join(OUTPUT_TABLES_PATH, "mlp_topology_comparison.csv"), index=False)




--- Running MLP Baselines (10 runs each) ---


## Getting result tables

In [7]:
# if already available, just load previous results
mlp_topology_comparison_df = pd.read_csv(os.path.join(OUTPUT_TABLES_PATH, "mlp_topology_comparison.csv"))
topology_comparison_df = pd.read_csv(os.path.join(OUTPUT_TABLES_PATH, "topology_comparison_df.csv"))

In [8]:
def get_paper_ready_table(df, group_by_col):
    # 1. Find best based on Mean F1
    idx = df.groupby(group_by_col)['F1'].idxmax()
    table = df.loc[idx].copy()
    
    # 2. Create "Mean ± Std" strings
    table['F1 Score'] = table.apply(lambda r: f"{r['F1']:.4f} ± {r['F1_std']:.4f}", axis=1)
    table['Accuracy Score'] = table.apply(lambda r: f"{r['Accuracy']:.4f} ± {r['Accuracy_std']:.4f}", axis=1)
    
    return table

# --- ARCHITECTURAL / MODEL COMPARISON ---
arch_table = get_paper_ready_table(mlp_topology_comparison_df, 'Model')
arch_table = arch_table[['Model', 'Graph', 'K', 'F1 Score', 'Accuracy Score']].sort_values(by='F1 Score', ascending=False)
print("\n--- BEST OF EACH ARCHITECTURE / MODEL (10-RUN AVG) ---")
print(arch_table.to_string(index=False))

# --- TOPOLOGY / CONFIGURATION COMPARISON ---
topo_table = get_paper_ready_table(mlp_topology_comparison_df, 'Graph')
topo_table = topo_table.rename(columns={'Graph': 'Topology / Baseline Configuration'})
topo_table = topo_table[['Topology / Baseline Configuration', 'Model', 'K', 'F1 Score', 'Accuracy Score']].sort_values(by='F1 Score', ascending=False)
print("\n--- BEST OF EACH TOPOLOGY / CONFIGURATION (10-RUN AVG) ---")
print(topo_table.to_string(index=False))

# --- DEDICATED BASELINES COMPARISON ---
baselines = ['Logistic Regression', 'MLP', 'Random Forest']
base_df = mlp_topology_comparison_df[mlp_topology_comparison_df['Model'].isin(baselines)].copy()
if not base_df.empty:
    base_df['Feature Set'] = base_df['Graph'].apply(lambda g: g.split('_')[-1] if '_' in g else g)
    base_df['F1 Score'] = base_df.apply(lambda r: f"{r['F1']:.4f} ± {r['F1_std']:.4f}", axis=1)
    base_df['Accuracy Score'] = base_df.apply(lambda r: f"{r['Accuracy']:.4f} ± {r['Accuracy_std']:.4f}", axis=1)
    base_table = base_df[['Model', 'Feature Set', 'F1 Score', 'Accuracy Score']].sort_values(by=['Model', 'Feature Set'])
    print("\n--- BASELINE MODELS COMPARISON (10-RUN AVG) ---")
    print(base_table.to_string(index=False))




--- BEST OF EACH ARCHITECTURE / MODEL (10-RUN AVG) ---
              Model           Graph  K        F1 Score  Accuracy Score
      Random Forest      RF_Spatial  0 0.6866 ± 0.0024 0.7051 ± 0.0023
    TransformerConv multirelational 27 0.6642 ± 0.0087 0.6754 ± 0.0082
               RGCN        temporal 29 0.6453 ± 0.0051 0.6522 ± 0.0070
                GAT multirelational 19 0.6012 ± 0.0249 0.6062 ± 0.0274
                MLP    MLP_Combined  0 0.5751 ± 0.0367 0.6039 ± 0.0404
Logistic Regression      LR_Spatial  0 0.5443 ± 0.0036 0.5560 ± 0.0030

--- BEST OF EACH TOPOLOGY / CONFIGURATION (10-RUN AVG) ---
Topology / Baseline Configuration               Model  K        F1 Score  Accuracy Score
                       RF_Spatial       Random Forest  0 0.6866 ± 0.0024 0.7051 ± 0.0023
                  multirelational     TransformerConv 27 0.6642 ± 0.0087 0.6754 ± 0.0082
                      RF_Combined       Random Forest  0 0.6536 ± 0.0046 0.6807 ± 0.0040
                         tempor

## Plotting Topology without features vs Baselines (MLP, Logistic Regression, Random Forest)


In [9]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import numpy as np
import os

def plot_wildfire_results(results_df, final_comparison_df, OUTPUT_FIGURES_PATH):
    if not os.path.exists(OUTPUT_FIGURES_PATH):
        os.makedirs(OUTPUT_FIGURES_PATH)

    # ==========================================
    # GLOBAL PAPER-READY STYLING
    # ==========================================
    plt.rcParams.update({
        'font.family': 'serif',
        'font.size': 18,
        'axes.labelsize': 22,
        'axes.titlesize': 22,
        'xtick.labelsize': 18,
        'ytick.labelsize': 18,
        'legend.fontsize': 16,
        'axes.linewidth': 2.0,
        'grid.linewidth': 1.0,
        'figure.dpi': 300,
        'savefig.bbox': 'tight'
    })
    sns.set_theme(style="whitegrid", font="serif")
    
    graph_order = ['spatial', 'temporal', 'combined', 'multirelational']
    architectures = ['RGCN', 'GAT', 'TransformerConv']
    
    main_palette = "viridis" 
    
    # ---------------------------------------------------------
    # 0. DATA PREPARATION (Extract Baselines: MLP, LR, RF)
    # Distinct, high-contrast palette:
    # MLP = Royal Purple (#7b1fa2), LR = Warm Amber (#e65100), RF = Forest Green (#2e7d32)
    # ---------------------------------------------------------
    def get_best_baseline(model_name, default_name):
        subset = final_comparison_df[final_comparison_df['Model'] == model_name]
        if not subset.empty:
            best = subset.sort_values(by='F1', ascending=False).iloc[0]
            return {
                'model': model_name,
                'f1': best['F1'],
                'std': best['F1_std'],
                'name': best['Graph'].replace('_', ' '),
                'available': True
            }
        return {'model': model_name, 'f1': 0.5, 'std': 0.0, 'name': default_name, 'available': False}

    best_mlp = get_best_baseline('MLP', 'MLP Baseline')
    best_lr = get_best_baseline('Logistic Regression', 'LR Baseline')
    best_rf = get_best_baseline('Random Forest', 'RF Baseline')

    baseline_styles = [
        (best_mlp, '#7b1fa2', '--', 'Best MLP'),
        (best_rf, '#2e7d32', '-.', 'Best RF'),
        (best_lr, '#e65100', ':', 'Best LR')
    ]

    # ---------------------------------------------------------
    # 1a. AVERAGE PERFORMANCE PER TOPOLOGY
    # ---------------------------------------------------------
    fig1, ax1 = plt.subplots(figsize=(13, 7))
    topo_avg = results_df.groupby(['Graph', 'Model']).agg({'F1': 'mean', 'F1_std': 'mean'}).reset_index()
    topo_avg['Graph'] = pd.Categorical(topo_avg['Graph'], categories=graph_order, ordered=True)
    topo_avg['Model'] = pd.Categorical(topo_avg['Model'], categories=architectures, ordered=True)
    topo_avg = topo_avg.sort_values(['Graph', 'Model'])

    sns.barplot(data=topo_avg, x='Graph', y='F1', hue='Model', palette=main_palette, ax=ax1, edgecolor='black', linewidth=1.2)
    
    for i, model_name in enumerate(architectures):
        model_data = topo_avg[topo_avg['Model'] == model_name].set_index('Graph').reindex(graph_order)
        x_coords = [rect.get_x() + rect.get_width() / 2 for rect in ax1.containers[i]]
        ax1.errorbar(x=x_coords, y=model_data['F1'], yerr=model_data['F1_std'], fmt='none', c='black', capsize=6, elinewidth=2)

    all_baseline_f1s = [b['f1'] for b, _, _, _ in baseline_styles if b['available']]
    for b_info, color, ls, label_prefix in baseline_styles:
        if b_info['available']:
            ax1.axhline(b_info['f1'], color=color, linestyle=ls, linewidth=2.5, label=label_prefix + " (" + b_info['name'] + ")")
            ax1.axhspan(b_info['f1'] - b_info['std'], b_info['f1'] + b_info['std'], color=color, alpha=0.10)
    
    min_val = min([topo_avg['F1'].min()] + all_baseline_f1s) if all_baseline_f1s else topo_avg['F1'].min()
    max_val = max([topo_avg['F1'].max()] + all_baseline_f1s) if all_baseline_f1s else topo_avg['F1'].max()
    ax1.set_ylim(min_val - 0.05, max_val + 0.1)
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(0.05))
    ax1.legend(title="Architecture / Baseline", loc='upper center', bbox_to_anchor=(0.5, 1.22), ncol=3, frameon=True)
    plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, 'avg_f1_comparison.png'))
    plt.close(fig1)

    # ---------------------------------------------------------
    # 1b. PEAK PERFORMANCE (Max F1 with K-labels)
    # ---------------------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(13, 7))
    topo_peak = results_df.loc[results_df.groupby(['Graph', 'Model'])['F1'].idxmax()].copy()
    topo_peak['Graph'] = pd.Categorical(topo_peak['Graph'], categories=graph_order, ordered=True)
    topo_peak['Model'] = pd.Categorical(topo_peak['Model'], categories=architectures, ordered=True)
    topo_peak = topo_peak.sort_values(['Graph', 'Model'])

    sns.barplot(data=topo_peak, x='Graph', y='F1', hue='Model', palette='magma', ax=ax2, edgecolor='black', linewidth=1.2)
    
    for i, model_name in enumerate(architectures):
        model_peak_data = topo_peak[topo_peak['Model'] == model_name].set_index('Graph').reindex(graph_order)
        x_coords = [rect.get_x() + rect.get_width() / 2 for rect in ax2.containers[i]]
        ax2.errorbar(x=x_coords, y=model_peak_data['F1'], yerr=model_peak_data['F1_std'], fmt='none', c='black', capsize=5, elinewidth=1.8)

        for x, y, err, k in zip(x_coords, model_peak_data['F1'], model_peak_data['F1_std'], model_peak_data['K']):
            if not np.isnan(y):
                ax2.text(x, y + err + 0.008, "K=" + str(int(k)), ha='center', va='bottom', fontsize=14, fontweight='bold')

    for b_info, color, ls, label_prefix in baseline_styles:
        if b_info['available']:
            ax2.axhline(b_info['f1'], color=color, linestyle=ls, linewidth=2.5, label=label_prefix + " (" + b_info['name'] + ")")
            ax2.axhspan(b_info['f1'] - b_info['std'], b_info['f1'] + b_info['std'], color=color, alpha=0.10)
    
    ax2.set_ylim(min_val - 0.05, max_val + 0.12)
    ax2.legend(title="Architecture / Baseline", loc='upper center', bbox_to_anchor=(0.5, 1.22), ncol=3)
    plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, 'peak_f1_comparison.png'))
    plt.close(fig2)

    # ---------------------------------------------------------
    # 2. PERFORMANCE LIFT (Best GNN Peak vs Baselines)
    # ---------------------------------------------------------
    fig3, ax3 = plt.subplots(figsize=(10, 8))
    best_gnn = topo_peak.sort_values(by='F1', ascending=False).iloc[0]
    
    comp_models = []
    if best_lr['available']:
        comp_models.append({'label': "Best LR\n(" + best_lr['name'] + ")", 'f1': best_lr['f1'], 'std': best_lr['std'], 'color': '#e65100'})
    if best_mlp['available']:
        comp_models.append({'label': "Best MLP\n(" + best_mlp['name'] + ")", 'f1': best_mlp['f1'], 'std': best_mlp['std'], 'color': '#7b1fa2'})
    if best_rf['available']:
        comp_models.append({'label': "Best RF\n(" + best_rf['name'] + ")", 'f1': best_rf['f1'], 'std': best_rf['std'], 'color': '#2e7d32'})
    comp_models.append({'label': "Best " + best_gnn['Model'] + "\n(" + best_gnn['Graph'] + ")", 'f1': best_gnn['F1'], 'std': best_gnn['F1_std'], 'color': '#1976d2'})

    lift_df = pd.DataFrame(comp_models)
    sns.barplot(data=lift_df, x='label', y='f1', hue='label', palette=lift_df['color'].tolist(), legend=False, ax=ax3, edgecolor='black', linewidth=1.2)
    ax3.errorbar(x=range(len(lift_df)), y=lift_df['f1'], yerr=lift_df['std'], fmt='none', c='black', capsize=12, elinewidth=2.5)
    
    for i in range(len(comp_models) - 1):
        baseline_f1 = comp_models[i]['f1']
        lift_val = ((best_gnn['F1'] - baseline_f1) / baseline_f1) * 100
        y_pos = (baseline_f1 + best_gnn['F1']) / 2 if baseline_f1 < best_gnn['F1'] else baseline_f1 / 2
        ax3.annotate(f"Lift: {lift_val:+.1f}%", xy=(i, y_pos), 
                     ha='center', fontsize=15, fontweight='bold', color='white',
                     bbox=dict(boxstyle="round,pad=0.3", fc="#333333", ec="black", lw=1))

    ax3.set_ylabel("Macro F1 Score")
    ax3.set_xlabel("Model")
    ax3.set_ylim(min(lift_df['f1'].min() - 0.05, 0.45), max(lift_df['f1'].max() + 0.1, 0.75))
    plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, 'topology_performance_lift.png'))
    plt.close(fig3)

    # ---------------------------------------------------------
    # 2b. BASELINE MODELS COMPARISON ACROSS FEATURE SETS
    # ---------------------------------------------------------
    baseline_subset = final_comparison_df[final_comparison_df['Model'].isin(['Logistic Regression', 'MLP', 'Random Forest'])].copy()
    if not baseline_subset.empty:
        fig_base, ax_base = plt.subplots(figsize=(10, 7))
        baseline_subset['Feature_Set'] = baseline_subset['Graph'].apply(lambda g: g.split('_')[-1] if '_' in g else g)
        feature_order = ['Spatial', 'Temporal', 'Combined']
        baseline_subset['Feature_Set'] = pd.Categorical(baseline_subset['Feature_Set'], categories=feature_order, ordered=True)
        baseline_subset['Model'] = pd.Categorical(baseline_subset['Model'], categories=['Logistic Regression', 'MLP', 'Random Forest'], ordered=True)
        baseline_subset = baseline_subset.sort_values(['Feature_Set', 'Model'])

        palette_baselines = {'Logistic Regression': '#e65100', 'MLP': '#7b1fa2', 'Random Forest': '#2e7d32'}
        sns.barplot(data=baseline_subset, x='Feature_Set', y='F1', hue='Model', palette=palette_baselines, ax=ax_base, edgecolor='black', linewidth=1.2)
        
        for idx, model_name in enumerate(['Logistic Regression', 'MLP', 'Random Forest']):
            sub_m = baseline_subset[baseline_subset['Model'] == model_name].set_index('Feature_Set').reindex(feature_order)
            x_coords = [rect.get_x() + rect.get_width() / 2 for rect in ax_base.containers[idx]]
            ax_base.errorbar(x=x_coords, y=sub_m['F1'], yerr=sub_m['F1_std'], fmt='none', c='black', capsize=5, elinewidth=1.8)

        ax_base.set_ylabel("Macro F1 Score")
        ax_base.set_xlabel("Feature Set (No Graph)")
        ax_base.set_title("Non-Graph Baselines Comparison Across Feature Sets", pad=15)
        ax_base.legend(title="Model", loc='upper left', frameon=True)
        ax_base.set_ylim(0.35, 0.75)
        plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, 'baseline_models_comparison.png'))
        plt.close(fig_base)

    # ---------------------------------------------------------
    # 3. SENSITIVITY HEATMAP
    # ---------------------------------------------------------
    m_data = results_df[results_df['Graph'] == 'multirelational']
    if not m_data.empty:
        fig4, ax4 = plt.subplots(figsize=(14, 6))
        pivot_heat = m_data.pivot_table(index="Model", columns="K", values="F1", aggfunc='mean')
        sns.heatmap(pivot_heat, annot=True, cmap="YlGnBu", fmt=".3f", 
                    linewidths=1, cbar_kws={'label': 'Mean F1'}, 
                    annot_kws={"size": 16, "weight": "bold"}, ax=ax4)
        ax4.set_title("Hyperparameter Sensitivity (K) on Multi-relational Graph", pad=20)
        plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, 'multirelational_heatmap.png'))
        plt.close(fig4)

    # ---------------------------------------------------------
    # 4. SENSITIVITY BAR PLOT
    # ---------------------------------------------------------
    if not m_data.empty:
        fig5, ax5 = plt.subplots(figsize=(14, 7))
        sns.barplot(data=m_data, x='K', y='F1', hue='Model', palette='tab10', ax=ax5, edgecolor='black')
        ax5.set_ylabel("Macro F1 Score")
        ax5.set_xlabel("Neighborhood Size (K)")
        ax5.legend(title="Architecture", loc='lower right', frameon=True)
        ax5.set_ylim(m_data['F1'].min() - 0.02, m_data['F1'].max() + 0.05)
        plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, 'sensitivity_barplot.png'))
        plt.close(fig5)
    
    plt.show()



In [10]:
plot_wildfire_results(topology_comparison_df, mlp_topology_comparison_df, OUTPUT_FIGURES_PATH)